In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
    


In [ ]:
import os
os.listdir(); os.chdir("/home/azm0269@auburn.edu/clover/")

from clover.utils.utils import notebook_line_magic
notebook_line_magic()
    


# B2-DiffuRL

Backward progressive training expands the optimized denoising suffix from the last few steps to the full trajectory. Branch-based sampling forks several continuations from the same suffix start, scores the terminal images, and trains on best/worst continuations from each branch group.
    


In [ ]:
from __future__ import annotations

import gc
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Callable

import torch
from PIL import Image
from torch import Tensor
from tqdm.auto import trange

from clover.utils.rewards_utils import aesthetic_proxy_reward

from clover.utils.baseline_utils import (
    backward_progressive_interval_length,
    decode_latents,
    ddpm_step_with_log_prob,
    encode_prompts,
    load_lora_pipeline,
    predict_noise,
    ppo_update,
    resolve_gpu_ids,
    sample_prompt_batch,
    save_image_grid_outputs,
    save_json,
    save_lora_weights,
    select_branch_extremes,
    set_seed,
    standard_eval_prompts,
    suffix_step_indices,
    trainable_parameters,
    unet_config,
)
    


In [ ]:
@dataclass
class B2DiffuRLConfig:
    """Configuration for B2-DiffuRL with the same SD/LoRA setup as DDPO and DPOK.

    Args:
        model_id: Hugging Face model identifier or local Stable Diffusion path.
        output_dir: Directory for checkpoints, generated images, and metrics.
        seed: Random seed shared by sampling and optimization.
        gpu_ids: Candidate CUDA device ids.
        use_data_parallel: Whether to wrap the UNet in ``torch.nn.DataParallel``.
        prompt: Primary prompt used in the standard evaluation set.
        negative_prompt: Negative prompt used for classifier-free guidance.
        train_prompts: Prompt pool sampled during training.
        height: Generated image height in pixels.
        width: Generated image width in pixels.
        num_inference_steps: Number of denoising steps per rollout.
        guidance_scale: Classifier-free guidance scale.
        eta: DDPM transition stochasticity scale.
        rollouts_per_epoch: Number of branch roots sampled per training epoch.
        branch_size: Number of continuations sampled from each branch point.
        initial_interval_steps: Number of final denoising steps trained in epoch one.
        train_epochs: Number of backward-progressive training epochs.
        ppo_epochs: Number of PPO passes over selected branch pairs per epoch.
        minibatch_size: Minibatch size for selected branch continuations.
        learning_rate: AdamW learning rate.
        adam_epsilon: AdamW epsilon.
        lora_rank: LoRA adapter rank.
        lora_alpha: LoRA adapter alpha.
        lora_dropout: LoRA dropout probability.
        lora_target_modules: UNet module names receiving LoRA adapters.
        clip_range: PPO probability ratio clipping range.
        ppo_log_ratio_clip: Clamp applied before exponentiating log ratios.
        min_reward_gap: Minimum best/worst branch reward gap to train a pair.
        max_grad_norm: Gradient clipping norm.
        mixed_precision: Whether to use fp16 on CUDA.
        gradient_checkpointing: Whether to enable UNet gradient checkpointing.
        log_every: Epoch logging interval.
        save_every: Checkpoint/sample image interval.
    """

    model_id: str = "runwayml/stable-diffusion-v1-5"
    output_dir: str = "outputs/b2diffurl"
    seed: int = 17
    gpu_ids: list[int] = field(default_factory=lambda: [0, 1, 2])
    use_data_parallel: bool = False

    prompt: str = "a colorful clover field at sunrise, high detail"
    negative_prompt: str = "blurry, low quality, distorted"
    train_prompts: tuple[str, ...] = (
        "a colorful clover field at sunrise, high detail",
        "a close-up photo of a bright green clover leaf with dew",
        "a small robot holding a clover in a clean studio photo",
        "an impressionist painting of clovers under warm sunlight",
    )

    height: int = 512
    width: int = 512
    num_inference_steps: int = 30
    guidance_scale: float = 7.5
    eta: float = 1.0

    rollouts_per_epoch: int = 1
    branch_size: int = 3
    initial_interval_steps: int = 6
    train_epochs: int = 4
    ppo_epochs: int = 4
    minibatch_size: int = 1
    learning_rate: float = 1e-9
    adam_epsilon: float = 1e-4
    lora_rank: int = 2
    lora_alpha: int = 2
    lora_dropout: float = 0.0
    lora_target_modules: tuple[str, ...] = ("to_v",)
    clip_range: float = 1e-4
    ppo_log_ratio_clip: float = 2.0
    min_reward_gap: float = 0.0
    max_grad_norm: float = 0.1
    mixed_precision: bool = True
    gradient_checkpointing: bool = True

    log_every: int = 1
    save_every: int = 5


cfg = B2DiffuRLConfig()
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
save_json(Path(cfg.output_dir) / "config.json", asdict(cfg))
cfg
    


In [ ]:
gpu_ids = resolve_gpu_ids(cfg)
device = torch.device(f"cuda:{gpu_ids[0]}" if gpu_ids else "cpu")
dtype = torch.float16 if device.type == "cuda" and cfg.mixed_precision else torch.float32
generator = set_seed(cfg.seed, device)
print(device, dtype, f"gpu_ids={gpu_ids}")
    


In [ ]:
reward_fn: Callable[[list[Image.Image], list[str]], Tensor] = lambda images, prompts: aesthetic_proxy_reward(
    images,
    prompts,
    device=device,
)
    


In [ ]:
pipe = load_lora_pipeline(cfg, device=device, dtype=dtype, gpu_ids=gpu_ids)
lora_parameters = trainable_parameters(pipe.unet)
print(f"Training {sum(parameter.numel() for parameter in lora_parameters):,} LoRA parameters")
optimizer = torch.optim.AdamW(lora_parameters, lr=cfg.learning_rate, eps=cfg.adam_epsilon)
vae_scale_factor = 2 ** (len(pipe.vae.config.block_out_channels) - 1)
    


In [ ]:
@torch.no_grad()
def collect_branch_rollouts(
    pipe,
    batch_size: int,
    interval_steps: int,
) -> dict[str, Tensor | list[str] | list[Image.Image]]:
    """Collect branch-based rollouts for the current backward-progressive interval.

    Args:
        pipe: LoRA-adapted Stable Diffusion pipeline being optimized.
        batch_size: Number of branch roots to sample before forking continuations.
        interval_steps: Number of final denoising steps included in the active interval.

    Returns:
        A rollout dictionary containing selected suffix states, actions, old log
        probabilities, timesteps, terminal rewards, prompts, decoded images,
        selected branch-pair advantages, and branch reward-pair diagnostics.
    """
    if cfg.branch_size < 2:
        raise ValueError("cfg.branch_size must be at least 2 for branch-based sampling.")

    pipe.unet.eval()
    root_prompts = sample_prompt_batch(cfg.train_prompts, batch_size)
    root_prompt_embeds = encode_prompts(pipe, root_prompts, cfg.negative_prompt, device, dtype)
    pipe.scheduler.set_timesteps(cfg.num_inference_steps, device=device)
    timesteps = [int(t.item()) for t in pipe.scheduler.timesteps]
    step_indices = suffix_step_indices(len(timesteps), interval_steps)
    branch_start_idx = step_indices[0]

    latent_shape = (
        batch_size,
        unet_config(pipe).in_channels,
        cfg.height // vae_scale_factor,
        cfg.width // vae_scale_factor,
    )
    latents = torch.randn(latent_shape, generator=generator, device=device, dtype=dtype)
    latents = latents * pipe.scheduler.init_noise_sigma

    for prefix_idx in range(branch_start_idx):
        t = pipe.scheduler.timesteps[prefix_idx]
        timestep = timesteps[prefix_idx]
        noise_pred = predict_noise(pipe, latents, t, root_prompt_embeds, cfg.guidance_scale)
        latents, _ = ddpm_step_with_log_prob(pipe.scheduler, noise_pred, timestep, latents, generator, eta=cfg.eta)

    branch_prompts = [prompt for prompt in root_prompts for _ in range(cfg.branch_size)]
    branch_prompt_embeds = encode_prompts(pipe, branch_prompts, cfg.negative_prompt, device, dtype)
    latents = latents.repeat_interleave(cfg.branch_size, dim=0)
    branch_batch = len(branch_prompts)

    states, actions, log_probs, active_timesteps = [], [], [], []
    zero_rewards = torch.zeros(branch_batch, dtype=torch.float32)
    rewards = []
    for step_idx in step_indices:
        t = pipe.scheduler.timesteps[step_idx]
        timestep = timesteps[step_idx]
        states.append(latents.detach().float().cpu())
        noise_pred = predict_noise(pipe, latents, t, branch_prompt_embeds, cfg.guidance_scale)
        next_latents, log_prob = ddpm_step_with_log_prob(pipe.scheduler, noise_pred, timestep, latents, generator, eta=cfg.eta)
        actions.append(next_latents.detach().float().cpu())
        log_probs.append(log_prob.detach().float().cpu())
        active_timesteps.append(timestep)
        latents = next_latents
        rewards.append(zero_rewards.clone())

    images = decode_latents(pipe, latents)
    terminal_rewards = reward_fn(images, branch_prompts).detach().float().cpu()
    rewards[-1] = terminal_rewards
    selected_idx, selected_advantages, reward_pairs = select_branch_extremes(
        terminal_rewards,
        branch_size=cfg.branch_size,
        min_reward_gap=cfg.min_reward_gap,
    )
    selected_idx_cpu = selected_idx.cpu()
    selected_images = [images[i] for i in selected_idx_cpu.tolist()]
    selected_prompts = [branch_prompts[i] for i in selected_idx_cpu.tolist()]

    pipe.unet.train()
    return {
        "prompts": selected_prompts,
        "states": torch.stack(states, dim=1)[selected_idx_cpu],
        "actions": torch.stack(actions, dim=1)[selected_idx_cpu],
        "old_log_probs": torch.stack(log_probs, dim=1)[selected_idx_cpu],
        "timesteps": torch.tensor(active_timesteps, dtype=torch.long),
        "rewards": torch.stack(rewards, dim=1)[selected_idx_cpu],
        "advantages": selected_advantages.cpu(),
        "branch_reward_pairs": reward_pairs.cpu(),
        "images": selected_images,
        "all_images": images,
        "all_prompts": branch_prompts,
        "all_terminal_rewards": terminal_rewards,
        "interval_steps": interval_steps,
        "branch_start_idx": branch_start_idx,
    }
    


In [ ]:
history = []
for epoch in trange(1, cfg.train_epochs + 1):
    interval_steps = backward_progressive_interval_length(
        epoch=epoch,
        total_epochs=cfg.train_epochs,
        total_steps=cfg.num_inference_steps,
        initial_steps=cfg.initial_interval_steps,
    )
    rollout = collect_branch_rollouts(pipe, cfg.rollouts_per_epoch, interval_steps)
    metrics = ppo_update(
        pipe,
        rollout,
        optimizer,
        cfg,
        device,
        dtype,
        advantages=rollout["advantages"],
        reward_values=rollout["rewards"][:, -1],
    )
    pair_gaps = rollout["branch_reward_pairs"]
    metrics["mean_reward"] = metrics["reward_mean"]
    metrics["mean_pair_gap"] = float((pair_gaps[:, 0] - pair_gaps[:, 1]).mean().item()) if pair_gaps.numel() else float("nan")
    metrics["epoch"] = epoch
    metrics["interval_steps"] = interval_steps
    metrics["branch_start_idx"] = rollout["branch_start_idx"]
    history.append(metrics)

    if epoch % cfg.log_every == 0:
        print(metrics)

    if epoch % cfg.save_every == 0:
        ckpt_dir = Path(cfg.output_dir) / f"lora_epoch_{epoch:04d}"
        save_lora_weights(pipe, ckpt_dir)
        save_image_grid_outputs(
            rollout["images"],
            rollout["prompts"],
            Path(cfg.output_dir),
            f"epoch_{epoch:04d}_sample",
        )

    save_json(Path(cfg.output_dir) / "history.json", history)
    del rollout
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    


In [ ]:
rollout = collect_branch_rollouts(pipe, cfg.rollouts_per_epoch, cfg.num_inference_steps)
rollout["all_terminal_rewards"]
    


In [ ]:
@torch.no_grad()
def generate_eval_images(pipe, prompts: list[str], seed: int = 123) -> list[Image.Image]:
    """Generate deterministic evaluation images for the shared prompt set.

    Args:
        pipe: LoRA-adapted Stable Diffusion pipeline to evaluate.
        prompts: Prompt strings to render.
        seed: Random seed for deterministic evaluation sampling.

    Returns:
        A list of generated PIL images aligned with ``prompts``.
    """
    pipe.unet.eval()
    eval_generator = torch.Generator(device=device).manual_seed(seed)
    images = pipe(
        prompts,
        negative_prompt=[cfg.negative_prompt] * len(prompts),
        height=cfg.height,
        width=cfg.width,
        num_inference_steps=cfg.num_inference_steps,
        guidance_scale=cfg.guidance_scale,
        generator=eval_generator,
    ).images
    pipe.unet.train()
    return images


eval_prompts = standard_eval_prompts(cfg)
eval_images = generate_eval_images(pipe, eval_prompts)
eval_rewards = reward_fn(eval_images, eval_prompts).detach().float().cpu().tolist()
save_image_grid_outputs(eval_images, eval_prompts, Path(cfg.output_dir), "eval")
save_json(
    Path(cfg.output_dir) / "eval_metrics.json",
    [{"prompt": prompt, "reward": reward} for prompt, reward in zip(eval_prompts, eval_rewards)],
)

eval_images[0], eval_images[1], eval_images[2], eval_images[3]
    


In [ ]:
import matplotlib.pyplot as plt

images = eval_images
titles = eval_prompts

fig, axes = plt.subplots(2, 2, figsize=(8, 8))

for ax, img, title in zip(axes.ravel(), images, titles):
    ax.imshow(img)
    ax.set_title(title, size=10)
    ax.axis("off")

plt.tight_layout()
plt.show()
    


In [ ]:
final_dir = Path(cfg.output_dir) / "lora_final"
save_lora_weights(pipe, final_dir)
save_json(Path(cfg.output_dir) / "config.json", asdict(cfg))
save_json(Path(cfg.output_dir) / "history.json", history)
print(f"Saved fine-tuned LoRA weights to {final_dir}")
print(asdict(cfg))
    
